In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"bettyblankson","key":"8f8c0b68b42b30ccfca73e42b9547532"}'}

In [2]:
import os
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
!pip install kaggle -q

In [ ]:
data_dir = "./cinic10"
dataset = "mengcius/cinic10"

if not os.path.exists(data_dir):
    os.makedirs(data_dir, exist_ok=True)

# Download from Kaggle
!kaggle datasets download -d {dataset} -p {data_dir}

# Extract the zip file
import zipfile
archive_path = os.path.join(data_dir, "cinic10.zip")
with zipfile.ZipFile(archive_path, 'r') as zip_ref:
    zip_ref.extractall(data_dir)

print("CINIC-10 dataset is ready!")

Dataset URL: https://www.kaggle.com/datasets/mengcius/cinic10
License(s): unknown
 97% 729M/754M [00:04<00:00, 98.9MB/s]
100% 754M/754M [00:04<00:00, 177MB/s] 


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import seaborn as sns

In [ ]:
class SimpleDNN(nn.Module):
    def __init__(self, input_dim=32*32*3, hidden_sizes=[512,256,128], n_classes=10, dropout=0.4):
        super().__init__()
        layers = []
        in_dim = input_dim

        #Track which indices are the last two hidden layers
        last_two = {len(hidden_sizes)-2, len(hidden_sizes)-1}

        for i, h in enumerate(hidden_sizes):
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU(inplace=True))


            if i in last_two:
                layers.append(nn.Dropout(dropout))

            in_dim = h

        layers.append(nn.Linear(in_dim, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)

In [ ]:
def get_loaders(data_dir, batch_size=256):
    # CINIC-10 mean & std
    cinic_mean = [0.47889522, 0.47227842, 0.43047404]
    cinic_std  = [0.24205776, 0.23828046, 0.25874835]

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean, cinic_std)  ##Normalizing the input
    ])

    train_ds = datasets.ImageFolder(os.path.join(data_dir,'train'), transform=transform)
    val_ds   = datasets.ImageFolder(os.path.join(data_dir,'valid'), transform=transform)
    test_ds  = datasets.ImageFolder(os.path.join(data_dir,'test'), transform=transform)

    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2),
            DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2),
            DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2))


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()*imgs.size(0)
        _, preds = outputs.max(1)
        correct += (preds==labels).sum().item()
        total += imgs.size(0)
    return running_loss/total, correct/total

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()*imgs.size(0)
            _, preds = outputs.max(1)
            correct += (preds==labels).sum().item()
            total += imgs.size(0)
    return running_loss/total, correct/total



In [ ]:
def test_and_report(model, test_loader, device, class_names):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Accuracy
    acc = accuracy_score(all_labels, all_preds)
    print(f"Test Accuracy: {acc * 100:.2f}%")

    # Confusion matrix
    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    import matplotlib.pyplot as plt
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=False, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()


In [ ]:
!ls ./cinic10

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
train_loader, val_loader, test_loader = get_loaders(data_dir, batch_size=256)
class_names = train_loader.dataset.classes

model = SimpleDNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
best_val_acc = 0.0
epochs = 15

# Lists to store metrics
train_losses, val_losses = [], []
train_accs, val_accs = [], []



for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    print(f'Epoch {epoch+1}: train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}')
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_dnn_cinic.pth')
model.load_state_dict(torch.load('best_dnn_cinic.pth', map_location=device))
test_and_report(model, test_loader, device, class_names)

In [ ]:
# Plot Loss
plt.figure(figsize=(8,5))
plt.subplot(1,2,1)
plt.plot(range(1, epochs+1), train_losses, label='Train Loss', marker='o')
plt.plot(range(1, epochs+1), val_losses, label='Validation Loss',marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss over Epochs')
plt.legend()

# Plot Accuracy
plt.subplot(1,2,2)
plt.plot(range(1, epochs+1), train_accs, label='Train Accuracy', marker='o')
plt.plot(range(1, epochs+1), val_accs, label='Validation Accuracy', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()

plt.tight_layout()
plt.show()

# labels = [f"Class {i}" for i in range(10)]

# # Plot side by side
# fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# sns.heatmap(conf_mat, annot=False, fmt="d", cmap="Blues",
#             xticklabels=labels, yticklabels=labels, ax=axes[0])
# axes[0].set_title("Confusion Matrix - SNN")
# axes[0].set_xlabel("Predicted")
# axes[0].set_ylabel("True")